##### Install Necessary Libraries

In [1]:
!pip install sqlalchemy beautifulsoup4 selenium requests

#### Import Libraries

In [2]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
from sqlalchemy import create_engine


##### Test for response 200

In [3]:
url = "https://www.eyebuydirect.ca/eyeglasses/index.html"
response = requests.get(url)

print(f"Status Code: {response.status_code}")

if response.status_code == 200:
    print("Successfully accessed the webpage.")
else:
    print("Failed to access the webpage.")


Status Code: 200
Successfully accessed the webpage.


##### Configure Selenium Chromedriver options

In [10]:
options = webdriver.ChromeOptions()
options.headless = True
service = Service(executable_path=r'chromedriver-win64\chromedriver.exe')

#initiate the webdriver
driver = webdriver.Chrome(service=service, options=options)

#define the URL to scrape
url = "https://www.eyebuydirect.ca/eyeglasses/index.html"

# use selenium to open the webpage

# wait for the page to load
time.sleep(10)

# get the page source and close the browser
page_source = driver.page_source
driver.quit()


#### EXTRACTION LAYER

In [11]:
options = webdriver.ChromeOptions()
options.headless = True
service = Service(executable_path=r'chromedriver-win64\chromedriver.exe')

#initiate the webdriver
driver = webdriver.Chrome(service=service, options=options)



In [23]:
# List to store the extracted data
product_discounts = []
product_images = []
product_names = []
product_prices = []
product_colors = []
product_materials = []
product_shapes = []

# Assuming you know the total no of pages
total_pages = 49

for page_number in range(1, total_pages + 1):
    url = f"https://www.eyebuydirect.ca/eyeglasses/index.html?page={page_number}"
    driver = webdriver.Chrome(service=service, options=options)
    driver.get(url)
    time.sleep(60)  # Wait for the page to load

    # Now we use BeautifulSoup to parse the page source
    soup = BeautifulSoup(driver.page_source, 'html.parser')

    #Adjust the class selectors based on current website structure
    eyeglass = soup.find_all('div', class_='container_content-wrapper__EnCm1')

    for eyeglass in eyeglass:
        # Extracting the product_discounts
        product_discounts.append(eyeglass.find('div', class_='container_promo-desktop__rKzrT').text if eyeglass.find('div', class_='container_promo-desktop__rKzrT') else 'N/A')

        # Extracting the product_images
        product_images.append(eyeglass.find('img', class_='product-card_item-image__R4jMz').get('src') if eyeglass.find('img', class_='product-card_item-image__R4jMz') else 'N/A')

        # Extracting the product_names
        product_names.append(eyeglass.find('div', class_='product-card_item-name__2s8nP').text if eyeglass.find('div', class_='product-card_item-name__2s8nP') else 'N/A')

        # Extracting the product_prices
        product_prices.append(eyeglass.find('div', class_='product-card_item-price__2s8nP').text.replace('$', '').replace(',', '') if eyeglass.find('div', class_='product-card_item-price__2s8nP') else 'N/A')

        # Extracting the product_colors
        product_colors.append(eyeglass.find('div', class_='product-card_color-name__Ttw10').text if eyeglass.find('div', class_='product-card_color-name__Ttw10') else 'N/A')

        # Extracting the product_materials
        product_materials.append(eyeglass.find('span', class_='filter-panel_filter-title__Ktw_v').text if eyeglass.find('span', class_='filter-panel_filter-title__Ktw_v') else 'N/A')

        # Extracting the product_shapes
        product_shapes.append(eyeglass.find('span', class_='filter-panel_filter-title__Ktw_v').text if eyeglass.find('span', class_='filter-panel_filter-title__Ktw_v') else 'N/A')


        # Append the extracted data to the respective lists
        product_discounts = product_discounts.append(product_discounts)
        product_images = product_images.append(product_images)
        product_names = product_names.append(product_names) 
        product_prices = product_prices.append(product_prices)
        product_colors = product_colors.append(product_colors)
        product_materials = product_materials.append(product_materials)
        product_shapes = product_shapes.append(product_shapes)

        #once all the pages are extracted, quit the driver 
driver.quit()

# Create a DataFrame from the extracted data
data = {
    'product_discounts': product_discounts,
    'product_images': product_images,
    'product_names': product_names,
    'product_prices': product_prices,
    'product_colors': product_colors,
    'product_materials': product_materials,
    'product_shapes': product_shapes
}
df = pd.DataFrame(data)





        

In [24]:
# display the DataFrame
display(df.head())

,product_discounts,product_images,product_names,product_prices,product_colors,product_materials,product_shapes


In [26]:
df.shape

(0, 7)

In [27]:
# save the raw data to a CSV file
df.to_csv('eyebuydirect_eyeglasses.csv', index=False)


### Data Transformation


In [28]:
df.columns

Index(['product_discounts', 'product_images', 'product_names',
       'product_prices', 'product_colors', 'product_materials',
       'product_shapes'],
      dtype='str')

In [32]:
# product table
product_columns = ['product_names', 'product_prices', 'product_discounts']
product = df[product_columns].copy().drop_duplicates().reset_index(drop=True)

#create a product id
product.index.name = 'product_id'
product = product.reset_index()




In [33]:
product

,product_id,product_names,product_prices,product_discounts


In [34]:
#product_specifications table
product_specifications_columns = ['product_names','product_images', 'product_colors', 'product_materials', 'product_shapes']
product_specifications = df[product_specifications_columns].copy().drop_duplicates().reset_index(drop=True) 

product_specifications.index.name = 'specification_id'
product_specifications = product_specifications.reset_index()


In [35]:
product_specifications

,specification_id,product_names,product_images,product_colors,product_materials,product_shapes


In [36]:
# saving to csv files
product.to_csv('product.csv', index=False)
product_specifications.to_csv('product_specifications.csv', index=False)

### Loading Layer (to Pgadmin)

In [39]:
# define database connection parameters including the db name

db_params = {
    'username': 'postgres',
    'password': 'Debbie1234#',
    'host': 'localhost',
    'port': '5432',
    'database': 'smartereyewears'
}

#define the database connection url using the db parameters
db_url = f"postgresql://{db_params['username']}:{db_params['password']}@{db_params['host']}:{db_params['port']}/{db_params['database']}"

# create the db engine
!pip install sqlalchemy beautifulsoup4 selenium requests psycopg2-binary

# connect to the postgresql server
engine = create_engine(db_url)

with engine.connect() as connection:

    # load the product table into the database
    product.to_sql('product', con=connection, if_exists='replace', index=False)
    
    # load the product_specifications table into the database
    product_specifications.to_sql('product_specifications', con=connection, if_exists='replace', index=False)

    print("Data loaded successfully into the database.")

Data loaded successfully into the database.
